In [1]:
!pip install -q evaluate seqeval datasets==2.21.0 accelerate -U
!pip install --upgrade transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:0

In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [3]:
from datasets import load_dataset
import pandas as pd

In [4]:
ds = load_dataset("leduckhai/VietMed-NER")
ds = ds.remove_columns(["audio"]) 

Generating train split:   0%|          | 0/4616 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1154 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3497 [00:00<?, ? examples/s]

In [5]:
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

In [6]:
def convert_split(dataset_split):
    """
    Convert 1 split sang dạng mới: {"tokens": [...], "ner_tags": [...]}
    """
    result = []
    for row in dataset_split:
        tokens = row["words"]
        ner_tags = row["labels"]  # đã là list tag string rồi

        result.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })
    return result

train_converted = convert_split(ds["train"])
val_converted   = convert_split(ds["validation"])
test_converted  = convert_split(ds["test"])

# Chuyển sang pandas DataFrame nếu muốn
df_train = pd.DataFrame(train_converted)
df_val   = pd.DataFrame(val_converted)
df_test  = pd.DataFrame(test_converted)

print(df_train.head())

                                              tokens  \
0  [thì, cũng, giống, như, ba, má, mình, đã, từng...   
1  [cái, điều, thứ, hai, đó, là, đối, với, những,...   
2  [béo, phì, đó, cải, thiện, được, cái, chất, lư...   
3  [thì, huyết, khối, rất, dễ, hình, thành, vì, d...   
4  [hóa, và, chống, oxy, hóa, thì, nó, tác, động,...   

                                            ner_tags  
0  [0, 0, 0, 0, B-GENDER, B-GENDER, 0, 0, 0, 0, B...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, B-DRUGCHE...  
2  [B-DISEASESYMTOM, I-DISEASESYMTOM, 0, B-TREATM...  
3  [0, B-ORGAN, 0, 0, 0, 0, 0, 0, B-ORGAN, I-ORGA...  
4  [0, 0, 0, B-DRUGCHEMICAL, 0, 0, 0, 0, 0, 0, 0,...  


In [7]:
import json

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(train_converted, "train_vietmed_ner.jsonl")
save_jsonl(val_converted, "valid_vietmed_ner.jsonl")
save_jsonl(test_converted, "test_vietmed_ner.jsonl")

In [8]:
import pandas as pd
from collections import Counter

df = pd.read_json("/kaggle/working/train_vietmed_ner.jsonl", lines=True)

#jsonl → lines=True

# Ghép tất cả ner_tags thành 1 list lớn
all_tags = []
for tags in df["ner_tags"]:
    # Nếu tags là list số → chuyển thành tên
    if isinstance(tags[0], int):
        tags = [dataset.features["ner_tags"].feature.names[t] for t in tags if t != -100]
    all_tags.extend(tags)

# Đếm
count_df = pd.Series(all_tags).value_counts().reset_index()
count_df.columns = ["Label", "Count"]
count_df = count_df.sort_values("Label")

print("Tổng số token có label (không tính O và -100):", count_df["Count"].sum())
print(count_df.to_string(index=False))

Tổng số token có label (không tính O và -100): 111267
               Label  Count
                   0  87320
               B-AGE    455
          B-DATETIME    695
       B-DIAGNOSTICS    373
     B-DISEASESYMTOM   2966
      B-DRUGCHEMICAL   1127
         B-FOODDRINK    257
            B-GENDER    210
          B-LOCATION    292
B-MEDDEVICETECHNIQUE    327
        B-OCCUPATION    545
             B-ORGAN   1972
      B-ORGANIZATION     19
      B-PERSONALCARE    383
     B-PREVENTIVEMED    343
           B-SURGERY    200
    B-TRANSPORTATION      5
         B-TREATMENT    740
    B-UNITCALIBRATOR    822
               I-AGE    164
          I-DATETIME    911
       I-DIAGNOSTICS    388
     I-DISEASESYMTOM   3638
      I-DRUGCHEMICAL    895
         I-FOODDRINK    234
            I-GENDER     48
          I-LOCATION    292
I-MEDDEVICETECHNIQUE    396
        I-OCCUPATION    637
             I-ORGAN   1387
      I-ORGANIZATION     64
      I-PERSONALCARE    609
     I-PREVENTIVEMED  

In [9]:
from datasets import load_dataset
import json
from pathlib import Path

# === CẤU HÌNH CHỈ 1 NƠI ===
KEEP = {
    'O',
    'B-CAUSE', 'I-CAUSE',
    'B-DIAGNOSTICS', 'I-DIAGNOSTICS',
    'B-DISEASESYMTOM', 'I-DISEASESYMTOM',
    'B-DRUGCHEMICAL', 'I-DRUGCHEMICAL',
    'B-MEDDEVICETECHNIQUE', 'I-MEDDEVICETECHNIQUE',
    'B-ORGAN', 'I-ORGAN',
    'B-PERSONALCARE', 'I-PERSONALCARE',
    'B-PREVENTIVEMED', 'I-PREVENTIVEMED',
    'B-SURGERY', 'I-SURGERY',
    'B-TREATMENT', 'I-TREATMENT'
}

# Đường dẫn file gốc (bạn sửa lại nếu khác)
files = {
    "train": "/kaggle/working/train_vietmed_ner.jsonl",
    "val":   "/kaggle/working/valid_vietmed_ner.jsonl",    # sửa tên file thực tế
    "test":  "/kaggle/working/test_vietmed_ner.jsonl",    # sửa tên file thực tế
}

# Tự động tạo tên file output
output_files = {
    split: str(Path(path).with_name(Path(path).stem + "_21labels.jsonl"))
    for split, path in files.items()
}

print("Bắt đầu lọc 21 nhãn y khoa chất lượng cao cho train / val / test...\n")

for split, input_path in files.items():
    output_path = output_files[split]
    
    if not Path(input_path).exists():
        print(f"Cảnh báo: Không tìm thấy file: {input_path} → Bỏ qua {split}")
        continue
        
    count_kept = 0
    count_total = 0
    
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:
        
        for line in fin:
            ex = json.loads(line.strip())
            new_tags = []
            for tag in ex["ner_tags"]:
                count_total += 1
                if tag in KEEP:
                    new_tags.append(tag)
                    count_kept += 1
                else:
                    new_tags.append("O")
            ex["ner_tags"] = new_tags
            fout.write(json.dumps(ex, ensure_ascii=False) + "\n")
    
    kept_ratio = count_kept / count_total * 100 if count_total > 0 else 0
    print(f"{split.upper():5} → {output_path}")
    print(f"      Đã xử lý: {count_total:,} token → giữ lại {count_kept:,} ({kept_ratio:.1f}%)")
    print(f"      File sạch lưu tại: {output_path}\n")

print("HOÀN TẤT! Bạn đã có bộ dữ liệu 21 nhãn siêu sạch cho train/val/test")

Bắt đầu lọc 21 nhãn y khoa chất lượng cao cho train / val / test...

TRAIN → /kaggle/working/train_vietmed_ner_21labels.jsonl
      Đã xử lý: 111,267 token → giữ lại 17,330 (15.6%)
      File sạch lưu tại: /kaggle/working/train_vietmed_ner_21labels.jsonl

VAL   → /kaggle/working/valid_vietmed_ner_21labels.jsonl
      Đã xử lý: 27,657 token → giữ lại 4,231 (15.3%)
      File sạch lưu tại: /kaggle/working/valid_vietmed_ner_21labels.jsonl

TEST  → /kaggle/working/test_vietmed_ner_21labels.jsonl
      Đã xử lý: 77,675 token → giữ lại 10,037 (12.9%)
      File sạch lưu tại: /kaggle/working/test_vietmed_ner_21labels.jsonl

HOÀN TẤT! Bạn đã có bộ dữ liệu 21 nhãn siêu sạch cho train/val/test


In [10]:
# ==================== 1. LOAD 3 FILE ĐÃ LỌC 21 NHÃN ====================
from datasets import load_dataset, DatasetDict, Features, Sequence, Value, ClassLabel
import json
from pathlib import Path

# Danh sách 21 nhãn chính xác (theo thứ tự để tránh lỗi)
ALL_LABELS_21 = [
    'O',
    'B-CAUSE', 'I-CAUSE',
    'B-DIAGNOSTICS', 'I-DIAGNOSTICS',
    'B-DISEASESYMTOM', 'I-DISEASESYMTOM',
    'B-DRUGCHEMICAL', 'I-DRUGCHEMICAL',
    'B-MEDDEVICETECHNIQUE', 'I-MEDDEVICETECHNIQUE',
    'B-ORGAN', 'I-ORGAN',
    'B-PERSONALCARE', 'I-PERSONALCARE',
    'B-PREVENTIVEMED', 'I-PREVENTIVEMED',
    'B-SURGERY', 'I-SURGERY',
    'B-TREATMENT', 'I-TREATMENT'
]

# Định nghĩa features với ClassLabel đúng 21 nhãn
features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=ALL_LABELS_21))
})

# Load 3 file đã lọc
data_files = {
    "train": "/kaggle/working/train_vietmed_ner_21labels.jsonl",
    "validation": "/kaggle/working/valid_vietmed_ner_21labels.jsonl",
    "test": "/kaggle/working/test_vietmed_ner_21labels.jsonl",
}

dataset = load_dataset("json", data_files=data_files, features=features)

print(f"Train: {len(dataset['train'])} câu")
print(f"Val:   {len(dataset['validation'])} câu")
print(f"Test:  {len(dataset['test'])} câu")
print(f"Tổng nhãn: {len(ALL_LABELS_21)} nhãn")

# Tạo label2id, id2label
label2id = {label: i for i, label in enumerate(ALL_LABELS_21)}
id2label = {i: label for i, label in enumerate(ALL_LABELS_21)}

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train: 4616 câu
Val:   1154 câu
Test:  3497 câu
Tổng nhãn: 21 nhãn


In [11]:
MODEL_CHECKPOINT = "xlm-roberta-large"
OUTPUT_DIR = "viet-medical-ner-21labels-xlm-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(ALL_LABELS_21),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print("Model & tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

2025-12-11 15:13:49.586393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765466029.776299      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765466029.833072      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model & tokenizer loaded!


In [12]:
# ==================== 3. TOKENIZE + ALIGN LABELS ====================
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256,
        padding=False
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/4616 [00:00<?, ? examples/s]

Map:   0%|          | 0/1154 [00:00<?, ? examples/s]

Map:   0%|          | 0/3497 [00:00<?, ? examples/s]

In [13]:
# ==================== 4. DATA COLLATOR & METRICS ====================
from transformers import DataCollatorForTokenClassification
import evaluate
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=30,                    # Để dư không gian cho early stopping
    
    weight_decay=0.01,
    load_best_model_at_end=True,            # Quan trọng: load lại best model khi kết thúc
    metric_for_best_model="f1",
    greater_is_better=True,
    
    fp16=True,
    report_to=[],
    save_total_limit=1,                     # Giữ best + 1 checkpoint gần nhất
    logging_steps=50,
    
    # Cosine scheduler + warmup (như bạn muốn)
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",             # <--- Cosine ở đây
    
    seed=42,
)

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,          
            early_stopping_threshold=0.001     
        )
    ],
)

print("Bắt đầu training...")
trainer.train()

/tmp/ipykernel_20/907279548.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Bắt đầu training...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.680400,0.269611,0.515533,0.592950,0.551538,0.926709
2,0.193700,0.149586,0.712963,0.743602,0.727960,0.959432
3,0.123700,0.116156,0.770712,0.831000,0.799721,0.966952
4,0.085500,0.112939,0.789905,0.838725,0.813583,0.968507
5,0.064900,0.114895,0.847950,0.859005,0.853442,0.975268
6,0.042900,0.097451,0.844331,0.877354,0.860526,0.975992
7,0.031800,0.122957,0.840218,0.893771,0.866168,0.974798
8,0.025800,0.118804,0.860724,0.895220,0.877633,0.977112
9,0.016000,0.129057,0.862709,0.898117,0.880057,0.976859
10,0.014900,0.114261,0.865153,0.901497,0.882951,0.978414


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/di

TrainOutput(global_step=3625, training_loss=0.07586881923829687, metrics={'train_runtime': 4335.6662, 'train_samples_per_second': 31.94, 'train_steps_per_second': 1.003, 'total_flos': 7743245068589952.0, 'train_loss': 0.07586881923829687, 'epoch': 25.0})

In [15]:
print("Đánh giá trên tập test:")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(f"Test F1: {test_results['eval_f1']:.4f}")

trainer.save_model(OUTPUT_DIR)
print(f"Model đã lưu tại: {OUTPUT_DIR}")

from transformers import pipeline
ner = pipeline("ner", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, aggregation_strategy="simple")

test_sentence = "Bệnh nhân bị đau bụng dưới, sốt cao, dùng paracetamol và ciprofloxacin nhưng không giảm."
print(ner(test_sentence))

Đánh giá trên tập test:


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test F1: 0.6107
Model đã lưu tại: viet-medical-ner-21labels-xlm-roberta


The tokenizer you are loading from 'viet-medical-ner-21labels-xlm-roberta' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


[{'entity_group': 'DISEASESYMTOM', 'score': 0.99944377, 'word': 'đau bụng', 'start': 13, 'end': 21}, {'entity_group': 'DISEASESYMTOM', 'score': 0.99993676, 'word': 'sốt', 'start': 28, 'end': 31}, {'entity_group': 'DRUGCHEMICAL', 'score': 0.99897695, 'word': 'paracetamol', 'start': 42, 'end': 53}, {'entity_group': 'DRUGCHEMICAL', 'score': 0.9987097, 'word': 'ciprofloxacin', 'start': 57, 'end': 70}]


In [16]:
import os
import shutil

folder_path = "/kaggle/working/viet-medical-ner-21labels-xlm-roberta" 

if os.path.exists(folder_path):
    for item in os.listdir(folder_path):
        item_path = os.path.join(folder_path, item)
        
        if os.path.isdir(item_path) and "checkpoint" in item:
            print(f"Giữ lại folder: {item}")
            continue
            
        if item.endswith(".jsonl"):
            print(f"Giữ lại file data: {item}")
            continue
            
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
                print(f"-> Đã xóa file rác: {item}")
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"-> Đã xóa folder rác: {item}")
        except Exception as e:
            print(f"Lỗi khi xóa {item}: {e}")
else:
    print(f"Không tìm thấy thư mục {folder_path}")

-> Đã xóa file rác: tokenizer.json
Giữ lại folder: checkpoint-2900
-> Đã xóa file rác: model.safetensors
-> Đã xóa file rác: sentencepiece.bpe.model
-> Đã xóa file rác: config.json
-> Đã xóa file rác: tokenizer_config.json
-> Đã xóa file rác: training_args.bin
-> Đã xóa file rác: special_tokens_map.json


In [17]:
test_sentences_list = [
    "Những đối tượng nào cần thận trọng khi sử dụng thuốc Cotrimstada forte?",
    "Tại sao không nên dùng thuốc Biviantac quá 2 tuần mà không có ý kiến bác sĩ?",
    "Thuốc Agifuros 40 mg được dùng trong trường hợp bệnh lý nào?",
    "Tại sao bệnh nhân không nên sử dụng thuốc Co-Diovan nếu họ đang chạy thận nhân tạo?",
    "Đối tượng nào không nên sử dụng thuốc Aspirin STELLA theo chống chỉ định của nhà sản xuất?",
    "Liệu Brudoxil có an toàn với phụ nữ cho con bú không?"
]

all_results = ner(test_sentences_list)
for sentence, result in zip(test_sentences_list, all_results):
    print("--------------------------------------------------")
    print(f"Câu: {sentence}")
    print("Entities được nhận dạng:")
    for entity in result:
        print(f"  - Entity: **{entity['word']}** (Type: {entity['entity_group']}, Score: {entity['score']:.2f})")

--------------------------------------------------
Câu: Những đối tượng nào cần thận trọng khi sử dụng thuốc Cotrimstada forte?
Entities được nhận dạng:
  - Entity: **thuốc** (Type: DRUGCHEMICAL, Score: 1.00)
  - Entity: **Cotrimstada forte** (Type: DRUGCHEMICAL, Score: 0.93)
--------------------------------------------------
Câu: Tại sao không nên dùng thuốc Biviantac quá 2 tuần mà không có ý kiến bác sĩ?
Entities được nhận dạng:
  - Entity: **thuốc** (Type: DRUGCHEMICAL, Score: 1.00)
  - Entity: **Biviantac** (Type: DRUGCHEMICAL, Score: 0.99)
--------------------------------------------------
Câu: Thuốc Agifuros 40 mg được dùng trong trường hợp bệnh lý nào?
Entities được nhận dạng:
  - Entity: **Thuốc** (Type: DRUGCHEMICAL, Score: 1.00)
  - Entity: **Agifuros 40 mg** (Type: DRUGCHEMICAL, Score: 0.89)
--------------------------------------------------
Câu: Tại sao bệnh nhân không nên sử dụng thuốc Co-Diovan nếu họ đang chạy thận nhân tạo?
Entities được nhận dạng:
  - Entity: **thuốc**